### Quickstart: Compare runs, choose a model, and deploy it to a rest API

In this quickstart, you will:
- Run a hyperparameter sweep on a training script

- Compare the results of the runs on the MLFLOW UI

- Choose the best run and register it as a model

- Deploy the model to a REST API

- Build a container image suitable for deployment to a cloud platform

In [17]:
import keras
import numpy as np
import pandas as pd
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import mlflow
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from mlflow.models import infer_signature

In [3]:
data=pd.read_csv(
"https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
sep=";")

In [4]:
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [5]:
train,test=train_test_split(data,test_size=0.25,random_state=42)

In [8]:
train_x=train.drop(["quality"],axis=1).values
train_y=train[["quality"]].values.ravel()

In [9]:
test_x=test.drop(["quality"],axis=1).values
test_y=test[["quality"]].values.ravel()

In [10]:
train_x,valid_x,train_y,valid_y=train_test_split(train_x,train_y,test_size=0.20,random_state=42)

In [11]:
signature=infer_signature(train_x,train_y)

In [37]:
def train_model(params,epochs,train_x,train_y,valid_x,valid_y,test_x,test_y):
    #Define the model architecture
    mean=np.mean(train_x,axis=0)
    var=np.var(train_x,axis=0)

    model=keras.models.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean,variance=var),
            keras.layers.Dense(64,activation="relu"),
            keras.layers.Dense(1)
        ]
    )

    #compile the model
    model.compile(optimizer=keras.optimizers.SGD(
        learning_rate=params["lr"],
        momentum=params["momentum"]),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()]
    )

    #Train the model with lr and momentum params with mlflow tracking
    with mlflow.start_run(nested=True):
        model.fit(train_x,train_y,validation_data=(valid_x, valid_y),epochs=epochs,batch_size=32)

        #Evaluate the model
        eval_result=model.evaluate(valid_x,valid_y,batch_size=64)

        eval_rmse=eval_result[1]

        #log the parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse",eval_rmse)

        #log the model
        mlflow.tensorflow.log_model(model,"model",signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

In [38]:
def objective(params):
    result=train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y
    )
    return result

In [39]:
space={
    "lr":hp.loguniform("lr",np.log(1e-5),np.log(1e-1)),
    "momentum":hp.uniform("momentum",0.0,1.0)
}

In [40]:
mlflow.set_experiment("/wine-quality")
with mlflow.start_run():
    trials=Trials()
    best=fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=5,
        trials=trials
    )

    #Fetch the details of the best run
    best_run=sorted(trials.results,key=lambda x:x["loss"])[0]

    #log the best parameters, loss and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse",best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"],"model",signature=signature)

    #print out the best parameters and corresponding loss
    print("Best Parameters:",best)
    print("Best eval loss:",best_run["loss"])

Epoch 1/3                                            

 1/92 ━━━━━━━━━━━━━━━━━━━━ 52s 582ms/step - loss: 33.4757 - root_mean_squared_error: 5.7858
29/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.7091 - root_mean_squared_error: 1.9259    
66/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.0197 - root_mean_squared_error: 1.4212
87/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.6785 - root_mean_squared_error: 1.2956
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 1.6233 - root_mean_squared_error: 1.2741 - val_loss: 0.5467 - val_root_mean_squared_error: 0.7394

Epoch 2/3                                            

 1/92 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 0.4047 - root_mean_squared_error: 0.6362
16/92 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.5167 - root_mean_squared_error: 0.7188 
34/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5215 - root_mean_squared_error: 0.7222
50/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5310 - root_mean_squared_error: 0.7287
67/92 ━━━━━━━━━━━━━━━━━━━━ 0s 

2026/07/12 19:35:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                     

 1/92 ━━━━━━━━━━━━━━━━━━━━ 47s 518ms/step - loss: 36.2166 - root_mean_squared_error: 6.0180
31/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 32.2703 - root_mean_squared_error: 5.6807   
63/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 31.9012 - root_mean_squared_error: 5.6481
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 31.3976 - root_mean_squared_error: 5.6034 - val_loss: 30.7549 - val_root_mean_squared_error: 5.5457

Epoch 2/3                                                                     

 1/92 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - loss: 30.3010 - root_mean_squared_error: 5.5046
25/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 30.4515 - root_mean_squared_error: 5.5183 
53/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 30.1865 - root_mean_squared_error: 5.4942
79/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 29.9924 - root_mean_squared_error: 5.4765
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 29.8673 - root

2026/07/12 19:35:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                     

 1/92 ━━━━━━━━━━━━━━━━━━━━ 51s 565ms/step - loss: 27.0898 - root_mean_squared_error: 5.2048
26/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 30.9469 - root_mean_squared_error: 5.5630   
53/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 30.9433 - root_mean_squared_error: 5.5627
78/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 30.8814 - root_mean_squared_error: 5.5571
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 30.8446 - root_mean_squared_error: 5.5538 - val_loss: 30.1790 - val_root_mean_squared_error: 5.4935

Epoch 2/3                                                                     

 1/92 ━━━━━━━━━━━━━━━━━━━━ 4s 50ms/step - loss: 29.1883 - root_mean_squared_error: 5.4026
19/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 29.9359 - root_mean_squared_error: 5.4714 
39/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 29.9450 - root_mean_squared_error: 5.4722
58/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 29.9887 - root

...

Epoch 1/3                                                                     

 1/92 ━━━━━━━━━━━━━━━━━━━━ 1:03 702ms/step - loss: 35.6326 - root_mean_squared_error: 5.9693
20/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 19.6812 - root_mean_squared_error: 4.4363    
36/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 12.7786 - root_mean_squared_error: 3.5747
55/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 9.3912 - root_mean_squared_error: 3.0645 
76/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 7.2767 - root_mean_squared_error: 2.6975
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 6.3064 - root_mean_squared_error: 2.5113 - val_loss: 1.6458 - val_root_mean_squared_error: 1.2829

Epoch 2/3                                                                     

 1/92 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - loss: 1.1143 - root_mean_squared_error: 1.0556
19/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.7440 - root_mean_squared_error: 1.3206 
39/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.5525 - root_mea

2026/07/12 19:36:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



100%|██████████| 5/5 [01:23<00:00, 16.65s/trial, best loss: 0.705123782157898]

2026/07/12 19:36:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best Parameters: {'lr': np.float64(0.04769370162040927), 'momentum': np.float64(0.7548106793787386)}
Best eval loss: 0.705123782157898


In [42]:
loaded_model=mlflow.pyfunc.load_model("models:/m-fd80cd62735546bb974c71a2a4052bca")
predictions=loaded_model.predict(test_x)

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [43]:
predictions

array([[5.9176083],
       [6.854883 ],
       [6.5440826],
       ...,
       [6.162158 ],
       [6.5305924],
       [5.804245 ]], shape=(1225, 1), dtype=float32)